In [10]:
# loding the library
import pandas as pd
import numpy as np
import joblib
from sklearn.neighbors import NearestNeighbors

In [11]:
df = pd.read_csv("../Datasets/amazon_feature_dataset.csv")

In [12]:
df.columns

Index(['name', 'main_category', 'sub_category', 'image', 'link', 'ratings',
       'no_of_ratings', 'discount_price', 'actual_price', 'combined_text',
       'discount_percentage'],
      dtype='object')

##### Now we will start the model tranning

In [13]:
# creating the knn model
knn_model = NearestNeighbors(n_neighbors=11,metric="cosine")

In [14]:
# lodingg the hybrid feature model
hybrid_features = joblib.load("../Models/hybrid_features.pkl")

In [15]:
print(type(hybrid_features))
print(hybrid_features.shape)

<class 'numpy.ndarray'>
(551585, 389)


In [16]:

np.isnan(hybrid_features).sum()

np.int64(0)

In [17]:
# now we will train the model
knn_model.fit(hybrid_features)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [19]:
# just checking is model run correctly or not
print(type(knn_model))

<class 'sklearn.neighbors._unsupervised.NearestNeighbors'>


In [21]:
#checking the model perameter
print(knn_model.get_params())

{'algorithm': 'auto', 'leaf_size': 30, 'metric': 'cosine', 'metric_params': None, 'n_jobs': None, 'n_neighbors': 11, 'p': 2, 'radius': 1.0}


In [ ]:
# Test the model on the first product
distances, indices = knn_model.kneighbors(
    hybrid_features[0].reshape(1, -1)
)

print("Distances:")
print(distances)
print("\nIndices:")
print(indices)

Distances:
[[0.         0.00094153 0.00129005 0.00160678 0.00168078 0.00220684
  0.00267893 0.00303614 0.00330861 0.00346384 0.00356949]]

Indices:
[[     0    747 243792 339881  23363      6      7    891 243817    919
  339989]]


In [25]:
# just checking is it recomending the product or not
recommended_indices = indices[0][1:]  # Skip the queried product
df.iloc[recommended_indices][["name", "main_category", "sub_category", "ratings", "discount_price"]]

,name,main_category,sub_category,ratings,discount_price
747,Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...,appliances,all appliances,4.2,10.404263
243792,Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...,appliances,heating & cooling appliances,4.2,10.404263
339881,Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...,appliances,kitchen & home appliances,4.2,10.404263
23363,Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...,home & kitchen,all home & kitchen,4.2,10.404263
6,Lloyd 1.0 Ton 3 Star Inverter Split Ac (5 In 1...,appliances,air conditioners,4.2,10.308953
7,Lloyd 1.5 Ton 5 Star Inverter Split Ac (5 In 1...,appliances,air conditioners,4.3,10.596410
891,Lloyd 1.0 Ton 3 Star Inverter Split Ac (5 In 1...,appliances,all appliances,4.2,10.308953
243817,Lloyd 1.0 Ton 3 Star Inverter Split Ac (5 In 1...,appliances,heating & cooling appliances,4.2,10.308953
919,Lloyd 1.5 Ton 5 Star Inverter Split Ac (5 In 1...,appliances,all appliances,4.3,10.596410
339989,Lloyd 1.0 Ton 3 Star Inverter Split Ac (5 In 1...,appliances,kitchen & home appliances,4.2,10.308953


##### Now we will create the recomendation Function

In [26]:
# first we will create the function 
def recommend_products(product_name, top_n=10):
    """Recommend similar products using the trained KNN model."""
    # Find the selected product
    product = df[df["name"] == product_name]

    # Check if the product exists
    if product.empty:
        print("Product not found!")
        return

    # Get product index
    product_index = product.index[0]

    # Find nearest neighbours
    distances, indices = knn_model.kneighbors(hybrid_features[product_index].reshape(1, -1))

    # Remove the queried product
    recommended_indices = indices[0][1:top_n + 1]

    # Get recommendations
    recommendations = df.iloc[recommended_indices][["name","main_category","sub_category","ratings","discount_price"]].copy()
    # Add similarity score

    recommendations["similarity_score"] = 1 - distances[0][1:top_n + 1]

    return recommendations

##### Observation
- A reusable recommendation function was created.
- It retrieves the most similar products using the trained KNN mode

##### Now we test the recomendation function

In [29]:
# select the product
product_name = df.iloc[225]["name"]
print(product_name)

FIDEL Tafta Heavy Duty Air Conditioner Outdoor Ac stand for 1 Ton, 1.1 Ton, 1.2 Ton, 1.5 Ton, 2 Ton Outdoor Units (Pack of...


In [ ]:
# generating the recomendation
recommend_products(product_name)

,name,main_category,sub_category,ratings,discount_price,similarity_score
247817,FIDEL Tafta Heavy Duty Air Conditioner Outdoor...,appliances,heating & cooling appliances,3.6,6.551080,0.985494
6960,DAIVE TAFTA Heavy Duty Air Conditioner Outdoor...,appliances,all appliances,3.5,6.475433,0.803862
244948,DAIVE TAFTA Heavy Duty Air Conditioner Outdoor...,appliances,heating & cooling appliances,3.5,6.475433,0.801029
344717,DAIVE TAFTA Heavy Duty Air Conditioner Outdoor...,appliances,kitchen & home appliances,3.5,6.475433,0.793285
248855,AJAMI Heavy Duty Air Conditioner Outdoor Ac st...,appliances,heating & cooling appliances,3.6,6.551080,0.769910
248525,AJAMI Special Coated Super Quality Split Ac Ai...,appliances,heating & cooling appliances,3.6,6.551080,0.742428
371,"Godrej 1.5 Ton 3 Star Window AC (Copper, 2019 ...",appliances,air conditioners,3.5,6.522093,0.732678
277,Voltas 183 LZH 1.5 Ton 3 Star Window AC (Coppe...,appliances,air conditioners,3.5,6.522093,0.724531
252026,BOLIDE™ Tafta Heavy Duty Air Conditioner Outdo...,appliances,heating & cooling appliances,3.2,6.436150,0.724300
246274,TWONE TAFTA Heavy Duty Air Conditioner Outdoor...,appliances,heating & cooling appliances,3.8,6.516193,0.715947


##### Observation
- The recommendation function successfully returned the top similar products.

In [34]:
# we are testing an other product
product_name = df.iloc[5000]["name"]
print(product_name)

QPT by STARQ QT-30 Super Pressure Commercial Washer (3000HPW)


In [ ]:
# doing same generatin the recomendation
recommend_products(product_name)

,name,main_category,sub_category,ratings,discount_price,similarity_score
3370,QPT by STARQ 2500W High Pressure Washer 180-25...,appliances,all appliances,4.7,9.047821,0.980524
3325,QPT by STARQ 2500W High Pressure Washer 180-25...,appliances,all appliances,4.7,9.047821,0.980524
341907,QPT by STARQ 2500W High Pressure Washer 180-25...,appliances,kitchen & home appliances,4.7,9.047821,0.979102
508885,Electrolux 7.5kg 40°C Vapour Wash 5 Star EcoIn...,appliances,washing machines,5.0,10.448454,0.976381
508878,Electrolux 8kg/5kg 5 Star EcoInverter Fully Au...,appliances,washing machines,5.0,10.858826,0.975795
508869,Electrolux 9kg 5 Star UltraMix with 40°C Vapou...,appliances,washing machines,5.0,10.691740,0.975023
508916,Electrolux 8kg 5 Star UltraMix with 40°C Vapou...,appliances,washing machines,5.0,10.621108,0.973082
509565,Electrolux 9kg 5 Star 40°C Vapour Wash EcoInve...,appliances,washing machines,5.0,10.621108,0.968858
8568,ROCKWELL RVC550B Single glass door Visi cooler...,appliances,all appliances,4.8,9.928229,0.968779
8567,ROCKWELL RVC550B Single glass door Visi cooler...,appliances,all appliances,4.8,9.928229,0.968779
